In [8]:
from functools import partial
from notebooks._utils import report_accuracy_by_nparas
from notebooks._utils import calculate_accuracy, _calculate_accuracy

ds_name = "myriadlama"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

In [26]:
import os
from pyexpat import model
import pandas as pd

for model_name in ["llama3.2_1b", "llama3.2_1b_it", "qwen2.5_14b", "qwen2.5_14b_it"]:
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    report_accuracy = partial(
        report_accuracy_by_nparas, 
        dump_file_prefix=dump_file_prefix,
        single_para_qapair=True,
        explicit_prompts=False,
        repeat_paras=False, 
        num_fewshots=5)
    
    print("---- Calculating baseline ----")
    df = calculate_accuracy(
        dump_file_prefix=dump_file_prefix, 
        single_para_qapair=True, explicit_prompts=False, repeat_paras=False, 
        modifyattn=False, modifyrope=False, scale_score=0, 
        num_paraphrases=1, num_fewshots=5)


    print("---- ❌ attention mask / ❌ rope modifications / ❌ scale score ----")
    report_accuracy(modifyattn=False, modifyrope=False, scale_score=False)

    print("---- ✅ attention mask / ✅ rope modifications / ❌ scale score ----")
    report_accuracy(modifyattn=True, modifyrope=True, scale_score=False)

    

    logits_root = f"/home/y-guo/self-ensemble/self-ensemble/results/{model_name}"
    max_logits = os.path.join(logits_root, "syncedsample_ensemble_max-5paras-5samples.feather")
    avg_logits = os.path.join(logits_root, "syncedsample_ensemble_avg-5paras-5samples.feather")

    max_df = pd.read_feather(max_logits)
    avg_df = pd.read_feather(avg_logits)

    print("---- Logits-based Ensemble ----")
    _calculate_accuracy(max_df, "Logits-based Ensemble (max)")
    _calculate_accuracy(avg_df, "Logits-based Ensemble (avg)")


=================== Model: llama3.2_1b ===================
---- Calculating baseline ----
Acc: 0.3914 ==> 🏷️ 1paras 5shots 1QA      (Baseline)
---- ❌ attention mask / ❌ rope modifications / ❌ scale score ----
Acc: 0.3204 ==> 🏷️ 5paras 5shots 1QA     
---- ✅ attention mask / ✅ rope modifications / ❌ scale score ----
Acc: 0.4450 ==> 🏷️ 5paras 5shots 1QA   +Attn +Rope 
---- Logits-based Ensemble ----
Acc: 0.4524 ==> 🏷️ Logits-based Ensemble (max)
Acc: 0.4680 ==> 🏷️ Logits-based Ensemble (avg)

=================== Model: llama3.2_1b_it ===================
---- Calculating baseline ----
Acc: 0.3514 ==> 🏷️ 1paras 5shots 1QA      (Baseline)
---- ❌ attention mask / ❌ rope modifications / ❌ scale score ----
Acc: 0.3787 ==> 🏷️ 5paras 5shots 1QA     
---- ✅ attention mask / ✅ rope modifications / ❌ scale score ----
Acc: 0.3885 ==> 🏷️ 5paras 5shots 1QA   +Attn +Rope 
---- Logits-based Ensemble ----
Acc: 0.4018 ==> 🏷️ Logits-based Ensemble (max)
Acc: 0.4193 ==> 🏷️ Logits-based Ensemble (avg)

====

In [ ]:
_calculate_accuracy(max_df, "Logits-based Ensemble (max)")
_calculate_accuracy(avg_df, "Logits-based Ensemble (avg)")

Acc: 0.5872 ==> 🏷️ max_logits
Acc: 0.6141 ==> 🏷️ avg_logits
